# Mini-projet Apriori et FP-Growth

## Partie 1 — Préparation des ventes

Le CSV fourni contient les champs nécessaires pour construire les paniers et segmenter par pays ou par date. On utilise `StockCode` comme identifiant stable du produit, et `Description` pour rendre les exemples lisibles. Le fichier nettoyé conserve une ligne par ligne de vente retenue, avec son pays et sa date.

In [1]:
import csv
from collections import Counter
from datetime import datetime
from pathlib import Path

data_dir = Path("data") if Path("data").is_dir() else Path("../data")
source_path = data_dir / "transaction_dataset.csv"
clean_path = data_dir / "processed" / "transactions_clean.csv"

columns = ["InvoiceNo", "StockCode", "Description", "Quantity", "Country", "InvoiceDate"]
with source_path.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    missing_columns = set(columns) - set(reader.fieldnames or [])
    if missing_columns:
        raise ValueError(f"Colonnes manquantes : {sorted(missing_columns)}")
    print("Colonnes disponibles :", ", ".join(reader.fieldnames))
    first_row = next(reader)
    print("Première ligne :", {column: first_row[column] for column in columns})

Colonnes disponibles : InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country
Première ligne : {'InvoiceNo': '536365', 'StockCode': '85123A', 'Description': 'WHITE HANGING HEART T-LIGHT HOLDER', 'Quantity': '6', 'Country': 'United Kingdom', 'InvoiceDate': '12/1/2010 08:26'}


### Nettoyage

On retire les factures annulées (préfixe C), les lignes auxquelles il manque une des six valeurs retenues, les quantités non entières ou non positives, les dates invalides et les lignes de frais ou d’ajustement repérées dans les codes du fichier. On écarte aussi une facture entière si ses lignes restantes se contredisent sur le pays ou le jour. Les écarts d’heure au sein d’une facture sont conservés. Les dates du CSV source sont au format mois/jour/année. L’export utilise le format ISO pour faciliter la segmentation.

Dans ce fichier, l’identifiant client est facultatif pour une analyse par facture. Dans un panier, un produit ne compte qu’une fois, même si plusieurs unités ou lignes portent le même code.

In [2]:
# Codes observés correspondant à des frais, remises ou écritures diverses.
non_products = {"POST", "DOT", "M", "C2", "D", "S", "BANK CHARGES", "AMAZONFEE", "CRUK", "B"}
removed = Counter()
sales = []
invoice_metadata = {}
conflicting_invoices = set()
source_line_count = 0

with source_path.open(encoding="utf-8-sig", newline="") as source:
    for raw in csv.DictReader(source):
        source_line_count += 1
        invoice, code, description, quantity_text, country, date_text = (
            (raw[column] or "").strip() for column in columns
        )
        if invoice.upper().startswith("C"):
            removed["Facture annulée"] += 1
            continue
        if not all((invoice, code, description, quantity_text, country, date_text)):
            removed["Valeur manquante"] += 1
            continue
        if code.upper() in non_products:
            removed["Frais, remise ou ajustement"] += 1
            continue
        try:
            quantity = int(quantity_text)
            date = datetime.strptime(date_text, "%m/%d/%Y %H:%M").isoformat(sep=" ")
        except ValueError:
            removed["Date ou quantité invalide"] += 1
            continue
        if quantity <= 0:
            removed["Quantité non positive"] += 1
            continue

        metadata = (country, date.split()[0])
        if invoice_metadata.setdefault(invoice, metadata) != metadata:
            conflicting_invoices.add(invoice)
        sales.append((invoice, code, description, quantity, country, date))

clean_sales = [row for row in sales if row[0] not in conflicting_invoices]
removed["Pays ou date contradictoire dans la facture"] = len(sales) - len(clean_sales)
assert source_line_count == len(clean_sales) + sum(removed.values())

print(f"{source_line_count:,} lignes brutes → {len(clean_sales):,} lignes retenues")
for reason, count in removed.items():
    print(f"{reason} : {count:,}")

541,909 lignes brutes → 522,415 lignes retenues
Frais, remise ou ajustement : 2,322
Valeur manquante : 7,422
Facture annulée : 9,288
Quantité non positive : 462
Pays ou date contradictoire dans la facture : 0


### Construction des paniers et export

La liste `transactions` est l’entrée de la partie 2. Le dictionnaire `baskets` permet de retrouver le panier à partir du numéro de facture. Pour une analyse par segment, on peut filtrer le CSV nettoyé sur le pays ou la date avant de regrouper à nouveau les codes par facture.

In [3]:
baskets = {}
for invoice, code, description, quantity, country, date in clean_sales:
    # Le dictionnaire conserve un libellé par code et déduplique le panier.
    baskets.setdefault(invoice, {}).setdefault(code, description)

transactions = [list(products) for products in baskets.values()]
unique_products = {code for basket in transactions for code in basket}
assert all(basket and len(basket) == len(set(basket)) for basket in transactions)

clean_path.parent.mkdir(parents=True, exist_ok=True)
with clean_path.open("w", encoding="utf-8", newline="") as output:
    writer = csv.writer(output)
    writer.writerow(columns)
    writer.writerows(clean_sales)

print(f"{len(transactions):,} transactions et {len(unique_products):,} articles distincts.")
print(f"{clean_path} — {len(clean_sales):,} lignes exportées.")

19,942 transactions et 3,831 articles distincts.
..\data\processed\transactions_clean.csv — 522,415 lignes exportées.


In [4]:
from itertools import islice

for invoice, products in islice(baskets.items(), 3):
    country, date = invoice_metadata[invoice]
    print(f"Facture {invoice} — {country} — {date} — {len(products)} produits")
    for code, description in products.items():
        print(f"  {code} : {description}")

Facture 536365 — United Kingdom — 2010-12-01 — 7 produits
  85123A : WHITE HANGING HEART T-LIGHT HOLDER
  71053 : WHITE METAL LANTERN
  84406B : CREAM CUPID HEARTS COAT HANGER
  84029G : KNITTED UNION FLAG HOT WATER BOTTLE
  84029E : RED WOOLLY HOTTIE WHITE HEART.
  22752 : SET 7 BABUSHKA NESTING BOXES
  21730 : GLASS STAR FROSTED T-LIGHT HOLDER
Facture 536366 — United Kingdom — 2010-12-01 — 2 produits
  22633 : HAND WARMER UNION JACK
  22632 : HAND WARMER RED POLKA DOT
Facture 536367 — United Kingdom — 2010-12-01 — 12 produits
  84879 : ASSORTED COLOUR BIRD ORNAMENT
  22745 : POPPY'S PLAYHOUSE BEDROOM
  22748 : POPPY'S PLAYHOUSE KITCHEN
  22749 : FELTCRAFT PRINCESS CHARLOTTE DOLL
  22310 : IVORY KNITTED MUG COSY
  84969 : BOX OF 6 ASSORTED COLOUR TEASPOONS
  22623 : BOX OF VINTAGE JIGSAW BLOCKS
  22622 : BOX OF VINTAGE ALPHABET BLOCKS
  21754 : HOME BUILDING BLOCK WORD
  21755 : LOVE BUILDING BLOCK WORD
  21777 : RECIPE BOX WITH METAL HEART
  48187 : DOORMAT NEW ENGLAND
